# DL Masterclass 03: Weight Initialization Theory

Welcome to Notebook 03. You have defined a perfect neural architecture and chosen the optimal activation functions. You press run, and the network immediately outputs `NaN` (Not a Number) for every prediction.

Why? Because of how you initialized the internal weights at step zero.

In deep networks, the mathematical scaling of the initial random numbers dictates whether the network trains successfully, explodes to infinity, or compresses to zero before the first epoch even finishes.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer involves Kaiming He's 2015 breakthrough paper:

> *"He Initialization divides random weights by $\sqrt{\frac{2}{n_{in}}}$. Derive why the '2' is mathematically required specifically when using ReLU activations, but causes the network to explode if using Tanh?"*

---

## Table of Contents
1. **Symmetry Breaking:** Why `zeros` or `ones` guarantees failure.
2. **The Variance Problem:** Exploding vs. Vanishing Forward Passes.
3. **Xavier/Glorot Initialization:** For Sigmoid and Tanh.
4. **He/Kaiming Initialization:** For ReLU and its variants.


## 1. Symmetry Breaking

What happens if you initialize all weights in an MLP to exactly `0.0` or `0.1`?

*   **Prediction:** Every single neuron in a given hidden layer will receive the exact same inputs and multiply them by the exact same weights. 
*   **Result:** Every neuron outputs the exact same value.
*   **Backprop:** When the error flows backwards, every neuron will receive the exact same gradient update. 

This means all 10,000 neurons in your layer will learn the exact same feature. A network with 10,000 identical neurons is mathematically identical to a network with 1 neuron. You must initialize with **random noise** to "break symmetry," ensuring different neurons learn different features.

## 2. The Variance Problem (Explosion vs Collapse)

Consider a purely linear 50-layer MLP without activations.

Let $W \sim \mathcal{N}(0, \sigma^2)$ (weights are drawn from a normal distribution with mean 0 and variance $\sigma^2$).

When you multiply the input $X$ by the weight matrix $W$, the variance of the output expands by a factor of $n_{in} \cdot \sigma^2$ (where $n_{in}$ is the number of input nodes).

*   If $n_{in} \cdot \sigma^2 > 1.0$: The numbers get slightly bigger at each layer. After 50 multiplications, they explode to `Infinity` (NaN).
*   If $n_{in} \cdot \sigma^2 < 1.0$: The numbers get slightly smaller. After 50 multiplications, they collapse to `0.0`.

**The Mathematical Law of Initialization:** You must choose $\sigma^2$ such that $n_{in} \cdot \sigma^2 = 1.0$.
Therefore, the variance of your random weights must be exactly $\frac{1}{n_{in}}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Simulating 50 Layers of Matrix Math
# -----------------------------------------------------
np.random.seed(42)

def simulate_network(std_dev, n_layers=50, n_nodes=500):
    """Passes data through a 50-layer linear network."""
    x = np.random.randn(1000, n_nodes) # Input Batch
    
    layer_variances = []
    
    for i in range(n_layers):
        w = np.random.randn(n_nodes, n_nodes) * std_dev
        x = np.dot(x, w)
        layer_variances.append(np.var(x))
        
    return layer_variances

n_nodes = 500

# 1. Too Small (std=0.01)
var_small = simulate_network(0.01)
# 2. Too Large (std=1.00)
var_large = simulate_network(1.00)
# 3. Mathematically Perfect (std = sqrt(1 / 500))
var_perfect = simulate_network(np.sqrt(1 / n_nodes))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(var_small, color='blue')
axes[0].set_title("Var=0.01 (Instant Collapse to 0)")

axes[1].plot(var_large, color='red')
axes[1].set_title("Var=1.0 (Instant Explosion to Infinity)")

axes[2].plot(var_perfect, color='green')
axes[2].set_title("Var=sqrt(1/N) (Stable at 1.0)")
axes[2].set_ylim(0, 2)

plt.tight_layout()
plt.show()

## 3. Xavier / Glorot Initialization (For Tanh)

If we add `Tanh` to the network, the law changes slightly. 
Xavier Glorot (2010) proved mathematically that for Tanh (and Sigmoid), we need to maintain stability going *forward* ($n_{in}$) and *backward* ($n_{out}$).

The optimal variance he derived is the average of the two: $\frac{2}{n_{in} + n_{out}}$. 
Weights are drawn from $\mathcal{N}(0, \sqrt{\frac{2}{n_{in} + n_{out}}})$.

*(Note: By default in PyTorch Linear layers, if you don't use this, things might break for deep networks).* 

## 4. He / Kaiming Initialization (For ReLU)

In 2015, ResNet researchers realized that Xavier Initialization failed spectacularly when using ReLU.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does ReLU require a '2' in the numerator: $\sqrt{\frac{2}{n_{in}}}$?*

**A:** 
Remember the definition of ReLU: $\max(0, x)$. 
It physically truncates exactly half of the signal (the negative half). Therefore, the variance of a signal passing through ReLU is instantly cut in half.

If you use Xavier initialization (which assumes the signal retains its full variance), within 30 layers of ReLU, the variance collapses to zero. 

Kaiming He proved that to offset the destructive "half-cut" of ReLU, you must explicitly multiply the initial variance by **2** to double the energy going in. 

$\text{Variance} = \frac{1}{n_{in}} \times 2 = \frac{2}{n_{in}} \rightarrow \text{StdDev} = \sqrt{\frac{2}{n_{in}}}$

This tiny tweak allowed the training of 152-layer deep networks for the first time in history.